In [ ]:
prefix_path = '../..'

import sys
sys.path.append(prefix_path)

import logging
import os
import json
import torch
from torch.utils.data import DataLoader

from detection_baselines.detection_vl_uncertainty import (
    load_uncertainty_dataset, uncertainty_collate_fn,
    UncertaintyThresholdDetector, train_uncertainty_detector, eval_uncertainty_detector
)
from egh_vlm.utils import load_phd_dataset, split_stratified

In [ ]:
DATASET_PATH = f'{prefix_path}/data/phd/phd_base_qwen3_vl_2b.json'
IMG_FOLDER_PATH = f'{prefix_path}/data/phd/images'
FEATURES_PATH = f'{prefix_path}/data/phd/features/vl_uncertainty_phd_qwen3.pt'
OUTPUT_PATH = f'{prefix_path}/data/phd/evaluations/evaluation_vl_uncertainty_phd_qwen3.json'
LOGGING_PATH = f'{prefix_path}/data/logs/log_vl_uncertainty_phd_qwen3.txt'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
TRAIN_RATIO = 0.7

#### Load Dataset

In [ ]:
dataset = load_phd_dataset(
    dataset_path=DATASET_PATH,
    img_folder_path=IMG_FOLDER_PATH,
)
print(f'Length of dataset: {len(dataset)}')

features = load_uncertainty_dataset(FEATURES_PATH)
print(f'Length of features: {len(features)}')

In [ ]:
train_dataset, test_dataset = split_stratified(features, train_ratio=TRAIN_RATIO, random_state=42)
train_dataloader = DataLoader(
    train_dataset,
    batch_size=32,
    collate_fn=uncertainty_collate_fn,
    shuffle=True,
)
test_dataloader = DataLoader(
    test_dataset,
    batch_size=32,
    collate_fn=uncertainty_collate_fn,
    shuffle=True,
)

#### Experiment

In [ ]:
os.makedirs(os.path.dirname(LOGGING_PATH), exist_ok=True)
logging.basicConfig(filename=LOGGING_PATH, level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

detector = UncertaintyThresholdDetector()
train_result = train_uncertainty_detector(detector, train_dataloader)
eval_result = eval_uncertainty_detector(detector, test_dataloader)

logging.info(f'threshold={train_result["threshold"]:.4f}, train_f1={train_result["f1"]:.4f}')
logging.info(f'acc={eval_result["acc"]:.4f}, f1={eval_result["f1"]:.4f}, precision={eval_result["precision"]:.4f}, recall={eval_result["recall"]:.4f}, pr_auc={eval_result["pr_auc"]:.4f}')

result = {
    'threshold': train_result['threshold'],
    'acc':       eval_result['acc'],
    'f1':        eval_result['f1'],
    'precision': eval_result['precision'],
    'recall':    eval_result['recall'],
    'pr_auc':    eval_result['pr_auc'],
}

logger = logging.getLogger()
for handler in logger.handlers[:]:
    handler.close()
    logger.removeHandler(handler)

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
with open(OUTPUT_PATH, 'w') as f:
    json.dump(result, f, indent=2)

print(f'threshold={train_result["threshold"]:.4f} | acc={eval_result["acc"]:.4f}, f1={eval_result["f1"]:.4f}, precision={eval_result["precision"]:.4f}, recall={eval_result["recall"]:.4f}, pr_auc={eval_result["pr_auc"]:.4f}')